In [ ]:
#### MIRBI form (Characterizing past fire occurrence in longleaf pine ecosystems with the Mid-Infrared Burn Index and a Random Forest classifier)
## MIRBI=10×SWIR2−9.8×SWIR1+2 

In [8]:

import arcpy
from arcpy.sa import *
import os
arcpy.env.addOutputsToMap = False   # don’t add GP outputs to the Contents

arcpy.env.overwriteOutput = True  # <-- Add this line
import math


In [2]:
arcpy.CheckOutExtension("Spatial")


'CheckedOut'

In [35]:
#first check if all files in the same spatial coord system


input_folder = r"F:\VFT\VFT_remote_sensing\Landsat8_GSP"

# Dictionary to track different spatial references
spatial_refs = {}

# Process each raster
for raster_file in os.listdir(input_folder):
    if raster_file.endswith(".tif"):  # Process only TIFF files
        raster_path = os.path.join(input_folder, raster_file)
        
        # Get raster spatial reference
        desc = arcpy.Describe(raster_path)
        spatial_ref = desc.spatialReference.name  # Get name of spatial reference
        
        # Store in dictionary (count occurrences)
        if spatial_ref not in spatial_refs:
            spatial_refs[spatial_ref] = []
        spatial_refs[spatial_ref].append(raster_file)

# Print results
print("\n📌 Spatial Reference Check:")
if len(spatial_refs) == 1:
    print(f"✅ All rasters are in the same spatial reference: {list(spatial_refs.keys())[0]}")
else:
    print("⚠️ Different spatial references detected!")
    for ref, files in spatial_refs.items():
        print(f"🔹 {ref}: {len(files)} rasters")
        for f in files[:5]:  # Show up to 5 files for each spatial ref
            print(f"    - {f}")


📌 Spatial Reference Check:
✅ All rasters are in the same spatial reference: WGS_1984_UTM_Zone_17N


In [24]:
# if different spatial reference, convert
input_folder = r"F:\VFT\VFT_remote_sensing\Landsat8_Lej"
output_folder = r"F:\VFT\VFT_remote_sensing\Landsat8_Lej\reprojected"

target_spatial_ref = arcpy.SpatialReference(32618)  # UTM Zone 18N (Modify if needed)   # UTM Zone 17N (EPSG: 32617)  

# Process each raster
for raster_file in os.listdir(input_folder):
    if raster_file.endswith(".tif"):  # Process only TIFF files
        raster_path = os.path.join(input_folder, raster_file)
        
        # Get raster properties
        desc = arcpy.Describe(raster_path)
        current_spatial_ref = desc.spatialReference
        
        # Check if reprojection is needed
        if current_spatial_ref.factoryCode != target_spatial_ref.factoryCode:
            output_path = os.path.join(output_folder, f"Reprojected_{raster_file}")
            
            # Reproject raster
            arcpy.management.ProjectRaster(
                raster_path, output_path, target_spatial_ref,
                "BILINEAR", 30  # Resampling method & cell size (modify if needed)
            )
            print(f"✅ Reprojected: {raster_file} → {output_path}")
        else:
            output_path = os.path.join(output_folder, raster_file)  
            arcpy.CopyRaster_management(raster_path, output_path)
            print(f"✔️ Already in target CRS: {raster_file} ➞ Copied to {output_path}")
            
print("fin")
      
# go back to check for consistent projection

✔️ Already in target CRS: Landsat_LC08_014036_20241208.tif ➞ Copied to F:\VFT\VFT_remote_sensing\Landsat8_Lej\reprojected\Landsat_LC08_014036_20241208.tif
✔️ Already in target CRS: Landsat_LC08_014036_20250109.tif ➞ Copied to F:\VFT\VFT_remote_sensing\Landsat8_Lej\reprojected\Landsat_LC08_014036_20250109.tif
✔️ Already in target CRS: Landsat_LC08_014036_20250125.tif ➞ Copied to F:\VFT\VFT_remote_sensing\Landsat8_Lej\reprojected\Landsat_LC08_014036_20250125.tif
✔️ Already in target CRS: Landsat_LC08_014036_20250226.tif ➞ Copied to F:\VFT\VFT_remote_sensing\Landsat8_Lej\reprojected\Landsat_LC08_014036_20250226.tif
✔️ Already in target CRS: Landsat_LC08_014036_20250330.tif ➞ Copied to F:\VFT\VFT_remote_sensing\Landsat8_Lej\reprojected\Landsat_LC08_014036_20250330.tif
✔️ Already in target CRS: Landsat_LC08_014036_20250415.tif ➞ Copied to F:\VFT\VFT_remote_sensing\Landsat8_Lej\reprojected\Landsat_LC08_014036_20250415.tif
✔️ Already in target CRS: Landsat_LC08_014036_20250501.tif ➞ Copied to

✅ Reprojected: Landsat_LT05_015036_19960218.tif → F:\remote sensing fire detection\VFT remote sensing fire detection\Cro_additional_2\reprojected\Reprojected_Landsat_LT05_015036_19960218.tif
✅ Reprojected: Landsat_LT05_015036_19960305.tif → F:\remote sensing fire detection\VFT remote sensing fire detection\Cro_additional_2\reprojected\Reprojected_Landsat_LT05_015036_19960305.tif
✅ Reprojected: Landsat_LT05_015036_19960321.tif → F:\remote sensing fire detection\VFT remote sensing fire detection\Cro_additional_2\reprojected\Reprojected_Landsat_LT05_015036_19960321.tif
✅ Reprojected: Landsat_LT05_015036_19960422.tif → F:\remote sensing fire detection\VFT remote sensing fire detection\Cro_additional_2\reprojected\Reprojected_Landsat_LT05_015036_19960422.tif
✅ Reprojected: Landsat_LT05_015036_19960508.tif → F:\remote sensing fire detection\VFT remote sensing fire detection\Cro_additional_2\reprojected\Reprojected_Landsat_LT05_015036_19960508.tif
✅ Reprojected: Landsat_LT05_015036_19960524.t

In [6]:

### get unqiue values in a pixel (check for empty cell values)


import os, arcpy

input_folder = r"F:\VFT\VFT_remote_sensing\Landsat8_Cr"
B1 = arcpy.Point(663035, 3890743)

unique_values_Cr = set()  # <-- keep this as the set

for raster_file in sorted(os.listdir(input_folder)):
    if raster_file.lower().endswith(".tif"):
        raster_path = os.path.join(input_folder, raster_file)
        print(f"Processing {raster_file}...")

        # Get band 8 (QA_PIXEL) as a single-band raster
        qa_band = arcpy.sa.ExtractBand(raster_path, 8)

        # Query pixel value at the point (same coord system as raster)
        res = arcpy.management.GetCellValue(qa_band, f"{B1.X} {B1.Y}")
        val_str = res.getOutput(0)

        # Skip nodata/blank values
        if not val_str or val_str.lower() == "nodata":
            print("  -> NoData at the point, skipping.")
            continue

        # Convert and add to the set of uniques
        try:
            val_B1 = int(float(val_str))
            unique_values_B1.add(val_B1)
        except ValueError:
            print(f"  -> Non-integer value '{val_str}', skipping.")

print("\n📌 Unique QA_PIXEL values at B1:")
print(sorted(unique_values_B1))

Processing Landsat_LC08_016035_20241206.tif...
Processing Landsat_LC08_016035_20241222.tif...
Processing Landsat_LC08_016035_20250107.tif...
Processing Landsat_LC08_016035_20250123.tif...
Processing Landsat_LC08_016035_20250224.tif...
Processing Landsat_LC08_016035_20250312.tif...
Processing Landsat_LC08_016035_20250328.tif...
Processing Landsat_LC08_016035_20250413.tif...
Processing Landsat_LC08_016035_20250429.tif...
Processing Landsat_LC08_016035_20250515.tif...
Processing Landsat_LC08_016035_20250531.tif...
Processing Landsat_LC08_016035_20250616.tif...
Processing Landsat_LC08_016035_20250718.tif...
Processing Landsat_LC08_016035_20250803.tif...
Processing Landsat_LC08_016036_20241206.tif...
Processing Landsat_LC08_016036_20241222.tif...
Processing Landsat_LC08_016036_20250107.tif...
Processing Landsat_LC08_016036_20250123.tif...
Processing Landsat_LC08_016036_20250224.tif...
Processing Landsat_LC08_016036_20250312.tif...
Processing Landsat_LC08_016036_20250328.tif...
Processing La

In [16]:
## obsolete



input_folder = r"F:\VFT\VFT_remote_sensing\Landsat8_Lej\reprojected"
output_folder = r"F:\VFT\VFT_remote_sensing\Landsat8_Lej\filtered"

# Define the exact cloud-free QA value and coordinates (depends on sites)
CLOUD_FREE_QA = 21824  #for landsat 8-9
#CLOUD_FREE_QA = 5440 # for landsat 5 bit flag 6 -- clear
gsp_li = arcpy.Point(749137, 3774536)  
gsp_bi = arcpy.Point(748942, 3774918) 
IA = arcpy.Point(288256,3830171)
CH = arcpy.Point(328503,3848824)
CM = arcpy.Point(319321,3848871)
B1 = arcpy.Point(663035,3890740)
B2 = arcpy.Point(664053,3890735)
ME = arcpy.Point(273569,3836308)

# Process each raster
#for raster_file in os.listdir(input_folder):
     #if raster_file.endswith(".tif") and not raster_file.endswith(".tif.xml"):  # Only process .tif files
        #raster_path = os.path.join(input_folder, raster_file)
#for raster_file in os.listdir(input_folder):
    #print(f"Scanning: {raster_file}")  # 👈 debug
   # if raster_file.lower().endswith(".tif") and ".xml" not in raster_file.lower():
        # Extract QA_PIXEL band (assuming it’s band 3)
for raster_file in os.listdir(input_folder):
    if raster_file.endswith(".tif"):
        raster_path = os.path.join(input_folder, raster_file)
        
        qa_band = arcpy.sa.ExtractBand(raster_path, 8) # band number, modify if needed

        # Extract pixel values at the two points
        #value_IA = arcpy.GetCellValue_management(qa_band, f"{IA.X} {IA.Y}")
        value_CH_raw = arcpy.GetCellValue_management(qa_band, f"{CH.X} {CH.Y}")
        value_CM_raw = arcpy.GetCellValue_management(qa_band, f"{CM.X} {CM.Y}")
        #value_gsp_li = arcpy.GetCellValue_management(qa_band, f"{gsp_li.X} {gsp_li.Y}")
        #value_gsp_bi = arcpy.GetCellValue_management(qa_band, f"{gsp_bi.X} {gsp_bi.Y}")
        #value_B1_raw = arcpy.GetCellValue_management(qa_band, f"{B1.X} {B1.Y}").getOutput(0)
        #value_B2_raw = arcpy.GetCellValue_management(qa_band, f"{B2.X} {B2.Y}").getOutput(0)
    

        if value_CH_raw == "NoData" or value_CM_raw == "NoData":
            print(f"❌ Skipped {raster_file} (NoData at either)")
            continue  # Skip this raster and move to the next one
        #import math
        value_CM = float(value_CM_raw)
        value_CH = float(value_CH_raw)
      
        if math.isnan(value_CM) or math.isnan(value_CH):
            print(f"❌ Skipped {raster_file} (NaN at either)")
            continue
# Keep only files where both values are exactly 21824
        if int(value_CH) == CLOUD_FREE_QA and int(value_CM) == CLOUD_FREE_QA:  # and value_CM == CLOUD_FREE_QA:
            output_path = os.path.join(output_folder, raster_file)
            arcpy.CopyRaster_management(raster_path, output_path)
    
            print(f"✅ Kept {raster_file} (cloud free)")
        else:
            print(f"❌ Skipped {raster_file} (cloud)")

print("🎉 Filtering complete!")

<class 'TypeError'>: float() argument must be a string or a real number, not 'Result'

In [36]:
# filter for cloud free images


input_folder = r"F:\VFT\VFT_remote_sensing\Landsat8_GSP"
output_folder = r"F:\VFT\VFT_remote_sensing\Landsat8_GSP\filtered"
os.makedirs(output_folder, exist_ok=True)

CLOUD_FREE_QA = 21824  # Landsat 8/9 clear

# points
CH = arcpy.Point(328503, 3848824)
CM = arcpy.Point(319321, 3848871)
CLOUD_FREE_QA = 21824  #for landsat 8-9
#CLOUD_FREE_QA = 5440 # for landsat 5 bit flag 6 -- clear
gsp_li = arcpy.Point(749137, 3774536)  
gsp_bi = arcpy.Point(748942, 3774918) 
IA = arcpy.Point(288256,3830171)
CH = arcpy.Point(328503,3848824)
CM = arcpy.Point(319321,3848871)
B1 = arcpy.Point(663035,3890740)
B2 = arcpy.Point(664053,3890735)
ME = arcpy.Point(273569,3836308)

for raster_file in os.listdir(input_folder):
    if not raster_file.lower().endswith(".tif"):
        continue

    raster_path = os.path.join(input_folder, raster_file)
    qa_band = arcpy.sa.ExtractBand(raster_path, 8)

    # --- read cell values (strings) ---
    gsp_li_str = arcpy.management.GetCellValue(qa_band, f"{gsp_li.X} {gsp_li.Y}").getOutput(0)
    gsp_bi_str = arcpy.management.GetCellValue(qa_band, f"{gsp_bi.X} {gsp_bi.Y}").getOutput(0)
    
    # --- handle NoData / blanks ---
   
    if gsp_li_str == "NoData"  or  gsp_bi_str == "NoData":
        print(f"❌ Skipped {raster_file} (NoData at either)")
        continue

    # --- convert to numbers safely ---
    try:
        gsp_li_str = int(float(gsp_li_str))  # handles "21824" and "21824.0"
        gsp_bi_str = int(float(gsp_bi_str))
    except ValueError:
        print(f"❌ Skipped {raster_file} (non-numeric value: CH={ch_str}, CM={cm_str})")
        continue

    # --- optional NaN guard (paranoia) ---
    if math.isnan(float(gsp_bi_str)) or math.isnan(float(gsp_li_str)):
        print(f"❌ Skipped {raster_file} (NaN at either)")
        continue

    # --- keep only cloud-free scenes at BOTH points ---
    if gsp_li_str == CLOUD_FREE_QA   and gsp_bi_str == CLOUD_FREE_QA:
        out_path = os.path.join(output_folder, raster_file)
        arcpy.management.CopyRaster(raster_path, out_path)
        print(f"✅ Kept {raster_file} (cloud free)")
    else:
        print(f"❌ Skipped {raster_file} (cloud)")

print("🎉 Filtering complete!")

✅ Kept Landsat_LC08_015036_20241215.tif (cloud free)
✅ Kept Landsat_LC08_015036_20241231.tif (cloud free)
✅ Kept Landsat_LC08_015036_20250116.tif (cloud free)
✅ Kept Landsat_LC08_015036_20250201.tif (cloud free)
✅ Kept Landsat_LC08_015036_20250217.tif (cloud free)
✅ Kept Landsat_LC08_015036_20250321.tif (cloud free)
❌ Skipped Landsat_LC08_015036_20250406.tif (cloud)
❌ Skipped Landsat_LC08_015036_20250422.tif (cloud)
❌ Skipped Landsat_LC08_015036_20250508.tif (cloud)
✅ Kept Landsat_LC08_015036_20250524.tif (cloud free)
✅ Kept Landsat_LC08_015036_20250609.tif (cloud free)
❌ Skipped Landsat_LC08_015036_20250625.tif (cloud)
❌ Skipped Landsat_LC08_015036_20250711.tif (cloud)
✅ Kept Landsat_LC08_015036_20250727.tif (cloud free)
🎉 Filtering complete!


In [37]:
input_folder = r"F:\VFT\VFT_remote_sensing\Landsat8_GSP\filtered"  
output_folder = r"F:\VFT\VFT_remote_sensing\Landsat8_GSP\MIRBI_filtered" 

arcpy.env.overwriteOutput = True  # <-- Add this line

for raster_file in os.listdir(input_folder):
    if raster_file.endswith(".tif"):  # Only process .tif files
        raster_path = os.path.join(input_folder, raster_file)

        # Define Raster and extract bands
        raster = Raster(raster_path)
        SWIR1 = arcpy.ia.ExtractBand(raster, band_ids=[6])  # band numbers representing SWIR2 and SWIR1
        SWIR2 = arcpy.ia.ExtractBand(raster, band_ids=[7])  # modify if needed

        # Apply MIRBI formula
        mirbi_raster = (10 * SWIR2) - (9.8 * SWIR1) + 2

        # Define output file path
        output_path = os.path.join(output_folder, f"MIRBI_{raster_file}")

        # Save the output
        mirbi_raster.save(output_path)
        print(f"Processed: {raster_file} → Saved: {output_path}")

print("✅ All MIRBI calculations completed!")

Processed: Landsat_LC08_015036_20241215.tif → Saved: F:\VFT\VFT_remote_sensing\Landsat8_GSP\MIRBI_filtered\MIRBI_Landsat_LC08_015036_20241215.tif
Processed: Landsat_LC08_015036_20241231.tif → Saved: F:\VFT\VFT_remote_sensing\Landsat8_GSP\MIRBI_filtered\MIRBI_Landsat_LC08_015036_20241231.tif
Processed: Landsat_LC08_015036_20250116.tif → Saved: F:\VFT\VFT_remote_sensing\Landsat8_GSP\MIRBI_filtered\MIRBI_Landsat_LC08_015036_20250116.tif
Processed: Landsat_LC08_015036_20250201.tif → Saved: F:\VFT\VFT_remote_sensing\Landsat8_GSP\MIRBI_filtered\MIRBI_Landsat_LC08_015036_20250201.tif
Processed: Landsat_LC08_015036_20250217.tif → Saved: F:\VFT\VFT_remote_sensing\Landsat8_GSP\MIRBI_filtered\MIRBI_Landsat_LC08_015036_20250217.tif
Processed: Landsat_LC08_015036_20250321.tif → Saved: F:\VFT\VFT_remote_sensing\Landsat8_GSP\MIRBI_filtered\MIRBI_Landsat_LC08_015036_20250321.tif
Processed: Landsat_LC08_015036_20250524.tif → Saved: F:\VFT\VFT_remote_sensing\Landsat8_GSP\MIRBI_filtered\MIRBI_Landsat_LC0

In [38]:
# calculate mean

arcpy.env.workspace = r"F:\VFT\VFT_remote_sensing\Landsat8_GSP\MIRBI_filtered"   # folder that contains MIBRI
arcpy.env.overwriteOutput = True  


mirbi_rasters = arcpy.ListRasters() 

# Compute mean pixel value across all rasters
mean_mirbi = arcpy.sa.CellStatistics(mirbi_rasters, "MEAN", "DATA")

# Save the output raster
output_raster = r"F:\VFT\VFT_remote_sensing\Landsat8_GSP\MIRBI_sumstats\Mean_MIRBI.tif"  # Change path as needed

mean_mirbi.save(output_raster)




In [39]:
# calculate sd

arcpy.env.workspace = r"F:\VFT\VFT_remote_sensing\Landsat8_GSP\MIRBI_filtered"
arcpy.env.overwriteOutput = True  


mirbi_rasters = arcpy.ListRasters() 

# Compute mean pixel value across all rasters
SD_mirbi = arcpy.sa.CellStatistics(mirbi_rasters, "STD", "DATA")

# Save the output raster
output_raster = r"F:\VFT\VFT_remote_sensing\Landsat8_GSP\MIRBI_sumstats\sd_MIRBI.tif"  # Change path as needed
SD_mirbi.save(output_raster)



In [41]:

input_folder = r"F:\VFT\VFT_remote_sensing\Landsat8_GSP\MIRBI_filtered"  
output_folder = r"F:\VFT\VFT_remote_sensing\Landsat8_GSP\sus_burn_GSPLI"  
mean_mirbi_path = r"F:\VFT\VFT_remote_sensing\Landsat8_GSP\MIRBI_sumstats\Mean_MIRBI.tif"
sd_mirbi_path = r"F:\VFT\VFT_remote_sensing\Landsat8_GSP\MIRBI_sumstats\sd_MIRBI.tif"

#gsp_li = arcpy.Point(749137, 3774536)  
#gsp_bi = arcpy.Point(748942, 3774918)
gsp_li = arcpy.Point(749137, 3774536)  
gsp_bi = arcpy.Point(748942, 3774918) 
IA = arcpy.Point(288256,3830171)
CH = arcpy.Point(328503,3848824)
CM = arcpy.Point(319321,3848871)
ME = arcpy.Point(273569,3836308)
B1 = arcpy.Point(663035,3890740)
B2 = arcpy.Point(664053,3890735)
for raster_file in os.listdir(input_folder):
    if raster_file.endswith(".tif"):  # Only process .tif files
        raster_path = os.path.join(input_folder, raster_file)
        raster = Raster(raster_path)
        
    #define variables
        mirbi_value = float(arcpy.GetCellValue_management(raster_path, f"{gsp_li.X} {gsp_li.Y}").getOutput(0)) #modify coord  # must define output as float
        mean_mirbi_value = float(arcpy.GetCellValue_management(mean_mirbi_path, f"{gsp_li.X} {gsp_li.Y}").getOutput(0))
        sd_mirbi_value = float(arcpy.GetCellValue_management(sd_mirbi_path, f"{gsp_li.X} {gsp_li.Y}").getOutput(0))
    
    # Compute absolute deviation from mean
        deviation = abs(mirbi_value - mean_mirbi_value)

  # select those not within 2 sd
        if deviation >= (2 * sd_mirbi_value):
            output_path = os.path.join(output_folder, os.path.basename(raster_path))
            arcpy.CopyRaster_management(raster_path, output_path)
            print(f"✅ Exported: {os.path.basename(raster_path)} (Outlier detected)")
        else:
            print(f"❌ Skipped: {os.path.basename(raster_path)} (Within normal range)")

❌ Skipped: MIRBI_Landsat_LC08_015036_20130402.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_015036_20130407.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_015036_20130421.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_015036_20130523.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_015036_20130710.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_015036_20130811.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_015036_20130827.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_015036_20130912.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_015036_20130928.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_015036_20131030.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_015036_20131217.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_015036_20140118.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_015036_20140219.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_015036_20140424.tif (Within normal

In [ ]:
# LC 8
GSP: 234 downloaded,  134 cloud free, 
Lej: